In [17]:
import json

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from pyproj import Transformer


# ============================================================
# 1. Daten laden
# ============================================================

df = pd.read_parquet("../data/mvv_realtime.parquet")
stops = pd.read_csv("../data/stops.txt")


# ============================================================
# 2. Stop IDs vereinheitlichen
# ============================================================

df["stop_id"] = df["stop_id"].astype(str)
stops["stop_id"] = stops["stop_id"].astype(str)


# ============================================================
# 3. Koordinaten der Haltestellen hinzufügen
# ============================================================

df = df.merge(
    stops[
        [
            "stop_id",
            "stop_lat",
            "stop_lon",
        ]
    ],
    on="stop_id",
    how="left",
)


# ============================================================
# 4. Gültige Beobachtungen
# ============================================================

df = df.dropna(
    subset=[
        "departure_delay",
        "stop_lat",
        "stop_lon",
    ]
).copy()


# ============================================================
# 5. Delay in Minuten
# ============================================================

df["delay_minutes"] = df["departure_delay"] / 60


# ============================================================
# 6. Durchschnittlicher Delay pro Haltestelle
# ============================================================

station_delay = (
    df.groupby(
        [
            "stop_id",
            "stop_name",
            "stop_lat",
            "stop_lon",
        ]
    )["delay_minutes"]
    .mean()
    .reset_index()
)


# ============================================================
# 7. GeoJSON laden
# ============================================================

with open("../data/munich.geojson", "r") as f:
    munich = json.load(f)


# ============================================================
# 8. Koordinatensystem transformieren
#
# GeoJSON: ETRS89 / UTM 32N
#          EPSG:25832
#
# GTFS:    WGS84
#          EPSG:4326
# ============================================================

transformer = Transformer.from_crs(
    "EPSG:25832",
    "EPSG:4326",
    always_xy=True,
)


def transform_coordinates(coordinates):
    """
    Transform UTM coordinates to longitude/latitude.
    """

    lon, lat = transformer.transform(
        coordinates[0],
        coordinates[1],
    )

    return [lon, lat]


# ============================================================
# 9. GeoJSON-Geometrie transformieren
# ============================================================

def transform_geometry(geometry):

    geometry_type = geometry["type"]
    coordinates = geometry["coordinates"]

    if geometry_type == "Polygon":

        return [
            [
                transform_coordinates(point)
                for point in ring
            ]
            for ring in coordinates
        ]

    elif geometry_type == "MultiPolygon":

        return [
            [
                [
                    transform_coordinates(point)
                    for point in ring
                ]
                for ring in polygon
            ]
            for polygon in coordinates
        ]

    else:
        raise ValueError(
            f"Unsupported geometry: {geometry_type}"
        )


# ============================================================
# 10. Karte erstellen
# ============================================================

fig, ax = plt.subplots(
    figsize=(12, 10)
)


# ============================================================
# 11. Stadtbezirke zeichnen
# ============================================================

for feature in munich["features"]:

    geometry = transform_geometry(
        feature["geometry"]
    )

    geometry_type = feature["geometry"]["type"]

    if geometry_type == "Polygon":

        for ring in geometry:

            xs = [point[0] for point in ring]
            ys = [point[1] for point in ring]

            ax.fill(
                xs,
                ys,
                alpha=0.15,
            )

            ax.plot(
                xs,
                ys,
                linewidth=0.7,
            )

    elif geometry_type == "MultiPolygon":

        for polygon in geometry:

            for ring in polygon:

                xs = [point[0] for point in ring]
                ys = [point[1] for point in ring]

                ax.fill(
                    xs,
                    ys,
                    alpha=0.15,
                )

                ax.plot(
                    xs,
                    ys,
                    linewidth=0.7,
                )


# ============================================================
# 12. Farbskala
# ============================================================

norm = Normalize(
    vmin=0,
    vmax=10,
)

cmap = plt.get_cmap(
    "RdYlGn_r"
)


# ============================================================
# 13. Haltestellen
# ============================================================

ax.scatter(
    station_delay["stop_lon"],
    station_delay["stop_lat"],
    c=station_delay["delay_minutes"],
    cmap=cmap,
    norm=norm,
    s=35,
    alpha=0.85,
    edgecolors="black",
    linewidths=0.3,
)


# ============================================================
# 14. Farbskala
# ============================================================

colorbar = plt.colorbar(
    ScalarMappable(
        norm=norm,
        cmap=cmap,
    ),
    ax=ax,
)

colorbar.set_label(
    "Average departure delay (minutes)"
)


# ============================================================
# 15. Titel
# ============================================================

ax.set_title(
    "Public Transport Delays in Munich",
    fontsize=16,
)


# ============================================================
# 16. Achsen
# ============================================================

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.set_aspect("equal")

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'pyproj'